# AdaptiveDepth Phase 4 — Colab / Kaggle
Runs the full-model baseline, block influence, compiled candidates, static baselines, resumable route benchmarking, and the go/no-go gate. Enable a GPU runtime before continuing. The notebook starts with a short smoke run; set `RUN_FULL = True` after it succeeds.

In [ ]:
import base64, os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle/working').exists() else '/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        if str(base).startswith('/content'):
            from google.colab import userdata
            return userdata.get(name)
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({
            'GIT_CONFIG_COUNT': '1',
            'GIT_CONFIG_KEY_0': 'http.extraHeader',
            'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}',
        })
    clone = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        env=clone_env, capture_output=True, text=True,
    )
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(
            f'Git clone failed: {detail}\n'
            'If the repository is private, add a read-only GITHUB_TOKEN to notebook Secrets '
            'and enable notebook access, then rerun this cell.'
        )
os.chdir(repo)
print('Repository:', repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch, transformers
assert torch.cuda.is_available(), 'Select a GPU runtime before running Phase 4'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)

In [ ]:
# Keep this False for the first run. Change it to True only after the smoke run succeeds.
RUN_FULL = False
config = 'configs/cloud_t4.yaml' if RUN_FULL else 'configs/cloud_smoke.yaml'
dataset = 'data/phase4_eval.jsonl' if RUN_FULL else 'data/smoke_eval.jsonl'
workspace = 'experiments/phase4' if RUN_FULL else 'experiments/phase4_smoke'
archive = 'phase4_artifacts.zip' if RUN_FULL else 'phase4_smoke_artifacts.zip'
print('Mode:', 'FULL' if RUN_FULL else 'SMOKE')
print('Workspace:', workspace)

command = [
    'adaptivedepth', 'run-phase4',
    '--config', config,
    '--dataset', dataset,
    '--compiler-config', 'configs/compiler.yaml',
    '--evaluation-config', 'configs/evaluation.yaml',
    '--workspace', workspace,
    '--resume',
]
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path
print((Path(workspace) / 'gate.md').read_text())
subprocess.run([sys.executable, '-m', 'zipfile', '-c', archive, workspace], check=True)
print('Download:', repo / archive)
if not RUN_FULL:
    print('Smoke run complete. Its gate verdict is diagnostic only.')
    print('Set RUN_FULL = True and rerun the final two cells for the full experiment.')